# Timonel SFT v4 — Prompt desde el archivo (no inline)**Objetivo:** reentrenar Llama 3.1 8B con el prompt CONTRATO.**Diferencias con v3:**- El prompt se lee del campo `prompt_limpio` del archivo. NO se reconstruye inline.- Así no hay desincronización entre dataset y notebook.**Prompt actual (contrato, sin regla):**- PROPOSITO: "La mision se considera completa cuando el barco esta a menos de 15 metros de la meta."- ACCIONES: "terminar: declara la mision como cumplida." (sin "Valida SOLO").**Referencia:** EXPEDIENTE/89 secciones 14, 15 y 16.

## 1. Instalar dependencias

In [ ]:
!pip install -q unsloth!pip install -q --no-deps trl peft accelerate bitsandbytes!pip install -q datasets

## 2. Cargar modelo

In [ ]:
from unsloth import FastLanguageModelimport torchmax_seq_length = 2048dtype = Noneload_in_4bit = Truemodel, tokenizer = FastLanguageModel.from_pretrained(    model_name = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",    max_seq_length = max_seq_length,    dtype = dtype,    load_in_4bit = load_in_4bit,)

## 3. Configurar LoRA

In [ ]:
model = FastLanguageModel.get_peft_model(    model,    r = 16,    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",                      "gate_proj", "up_proj", "down_proj"],    lora_alpha = 32,    lora_dropout = 0.0,    bias = "none",    use_gradient_checkpointing = "unsloth",    random_state = 3407,    use_rslora = False,    loftq_config = None,)

## 4. Cargar dataset

In [ ]:
from datasets import load_datasetimport jsonTRAIN_PATH = "/kaggle/input/datasets/ivnmielniczuk/timonel-cc003/train_cc003.jsonl"EVAL_PATH = "/kaggle/input/datasets/ivnmielniczuk/timonel-cc003/eval_cc003_confirmatorio.jsonl"dataset_train = load_dataset("json", data_files=TRAIN_PATH, split="train")eval_raw = [json.loads(l) for l in open(EVAL_PATH)]print(f"Train: {len(dataset_train)} | Eval: {len(eval_raw)}")

## 5. VERIFICACION

In [ ]:
REGLAS_PROHIBIDAS = [    "Valida SOLO", "valida solo",    "Usala si ", "Elegila si", "Usa terminar si", "Usa ir_a si",    "si estas a menos", "si estas mal alineado", "si estas bien alineado",]errores = 0for i, ej in enumerate(dataset_train):    user_content = ej["messages"][1]["content"]    for regla in REGLAS_PROHIBIDAS:        if regla.lower() in user_content.lower():            print(f"  [FAIL] train ej {i}: '{regla}'")            errores += 1for i, caso in enumerate(eval_raw):    prompt = caso.get("prompt_limpio", "")    for regla in REGLAS_PROHIBIDAS:        if regla.lower() in prompt.lower():            print(f"  [FAIL] eval ej {i}: '{regla}'")            errores += 1if errores > 0:    raise ValueError(f"ABORTADO: {errores} reglas de decision en prompts.")print(f">>> PROMPTS VERIFICADOS. Se puede entrenar. <<<")print()print("=== Primer prompt de train (recordatorio) ===")print(dataset_train[0]["messages"][1]["content"][:500])

## 6. Chat template

In [ ]:
from unsloth import get_chat_templatetokenizer = get_chat_template(tokenizer, chat_template="llama-3.1")def formatear(ejemplo):    texto = tokenizer.apply_chat_template(        ejemplo["messages"], tokenize=False, add_generation_prompt=False)    return {"text": texto}dataset_train = dataset_train.map(formatear)print("OK")

## 7. Entrenamiento

In [ ]:
from trl import SFTTrainerfrom transformers import TrainingArgumentsfrom unsloth import is_bfloat16_supportedtrainer = SFTTrainer(    model = model,    tokenizer = tokenizer,    train_dataset = dataset_train,    dataset_text_field = "text",    max_seq_length = max_seq_length,    dataset_num_proc = 2,    packing = False,    args = TrainingArguments(        per_device_train_batch_size = 2,        gradient_accumulation_steps = 4,        warmup_steps = 3,        max_steps = 60,        learning_rate = 2e-4,        fp16 = not is_bfloat16_supported(),        bf16 = is_bfloat16_supported(),        logging_steps = 5,        optim = "adamw_8bit",        weight_decay = 0.01,        lr_scheduler_type = "linear",        seed = 3407,        output_dir = "outputs_cc003_v4",        report_to = "none",    ),)stats = trainer.train()print(stats)

## 8. Guardar

In [ ]:
model.save_pretrained("timonel_lora_cc003_v4")tokenizer.save_pretrained("timonel_lora_cc003_v4")print("OK")!ls -la timonel_lora_cc003_v4/

## 9. Evaluacion (usa prompt_limpio del archivo)

In [ ]:
from collections import Counterimport math as _mFastLanguageModel.for_inference(model)def extraer_json(texto):    i = texto.find("{"); j = texto.rfind("}")    if i == -1 or j == -1 or j <= i: return None    try: return json.loads(texto[i:j+1])    except Exception: return Nonedef predecir(prompt_user):    messages = [        {"role": "system", "content": "Sos un agente de navegacion. Respondes EXCLUSIVAMENTE con JSON valido."},        {"role": "user", "content": prompt_user},    ]    inputs = tokenizer.apply_chat_template(        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")    with torch.no_grad():        out = model.generate(input_ids=inputs, max_new_tokens=64,                             temperature=0.0, do_sample=False, use_cache=True)    return tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True)aciertos = 0; total = 0uso = Counter(); por_accion_ok = Counter(); por_accion_tot = Counter()errores = []for caso in eval_raw:    state = caso["state"]    dx = state["meta_x"] - state["pos_x"]    dz = state["meta_z"] - state["pos_z"]    dist = _m.sqrt(dx*dx + dz*dz)    prompt_user = caso["prompt_limpio"]  # <-- del archivo    crudo = predecir(prompt_user)    parsed = extraer_json(crudo)    pred = parsed.get("accion") if parsed else "_error_"    true = caso["accion_optima"]    uso[pred] += 1    por_accion_tot[true] += 1    if pred == true:        aciertos += 1; por_accion_ok[true] += 1    else:        errores.append({"true": true, "pred": pred, "dist": round(dist,1), "familia": caso["familia"]})    total += 1print(f"=== RESULTADO VALIDACION CC003 v4 ===")print(f"Accuracy: {aciertos}/{total} = {100*aciertos/total:.1f}%")print(f"Uso de acciones: {dict(uso)}")print(f"Por accion: {dict(por_accion_ok)} / {dict(por_accion_tot)}")print()print(f"Errores: {len(errores)}")for e in errores[:15]:    print(f"  true={e['true']} pred={e['pred']} dist={e['dist']}m familia={e['familia']}")